# Layer 5: Cross-Exchange Correlation


In [ ]:
import os
import numpy as np
import pandas as pd

from mda.loader import load_trades
from mda.timestamps import add_ts_columns
from mda import EXCHANGES, DATA_DIR
from mda.layer1.rates import layer1_summary
from mda.layer5.cross_exchange import (
    compute_cross_venue_lag,
    compute_cumulative_capacity,
    detect_cascade_events,
)
from mda.plots.charts import (
    save_figure,
    plot_cross_venue_lag_cdf,
    plot_cumulative_capacity,
    plot_cascade_throughput,
)


In [ ]:
DATA_DIR = "/data/parquet"
REPORTS_DIR = "/data/reports"
os.makedirs(REPORTS_DIR, exist_ok=True)

df = load_trades(DATA_DIR, exchanges=EXCHANGES, add_ts_cols=True)
print("Trades shape:", df.shape)

results = layer1_summary(df)
rate_ts = results["rate_ts"]
rate_pcts = results["rate_pcts"]

print("\nRate percentile table:")
print(rate_pcts)


In [ ]:
lags = compute_cross_venue_lag(df)

print("Cross-venue lag summary (microseconds):")
print(lags.groupby(["exchange_a", "exchange_b"])["lag_us"].describe())


In [ ]:
# Plot X1: Cross-venue lag CDF
fig = plot_cross_venue_lag_cdf(lags)
fig.show()
save_figure(fig, "X1_cross_venue_lag_cdf", REPORTS_DIR)


In [ ]:
capacity = compute_cumulative_capacity(rate_pcts)

print("Cumulative capacity by number of exchanges:")
print(capacity)


In [ ]:
# Plot X2: Cumulative capacity
fig = plot_cumulative_capacity(capacity)
fig.show()
save_figure(fig, "X2_cumulative_capacity", REPORTS_DIR)


In [ ]:
cascade_windows = detect_cascade_events(df)
print(f"{len(cascade_windows)} cascade events detected")
if len(cascade_windows) > 0:
    print(cascade_windows.head())


In [ ]:
# Plot X3: Cascade throughput overlay on rate timeseries
fig = plot_cascade_throughput(rate_ts, cascade_windows)
fig.show()
save_figure(fig, "X3_cascade_throughput", REPORTS_DIR)
